> **Course source:** [llama-index/workflows.ipynb](https://huggingface.co/agents-course/notebooks/blob/cecf8edffb02101c90634b84076bd93b1a20c713/unit2/llama-index/workflows.ipynb).
> **Compatibility:** The local workflows run independently of Groq. The multi-agent example uses native function tools and verifies the calculator handoff and addition.

# Workflows in LlamaIndex


This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [1]:
COURSE_PACKAGES = ['llama-index-core==0.14.26', 'llama-index-llms-openai-like==0.7.2', 'llama-index-utils-workflow==0.11.0', 'pyvis==0.3.2']

import os
import subprocess
import sys

if not os.environ.get("HF_COURSE_BATCH"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                    "--disable-pip-version-check", *COURSE_PACKAGES], check=True, timeout=600)

And, let's log in to Hugging Face to use serverless Inference APIs.

In [2]:
import json
import os
from functools import lru_cache

import httpx
from openai import OpenAI

GROQ_BASE = "https://api.groq.com/openai/v1"
QWEN = "qwen/qwen3.8-27b"
GPT_OSS = ("openai/gpt-oss-20b", "openai/gpt-oss-120b")


def optional_secret(name):
    """Read a notebook-authorized secret without printing it."""
    if os.getenv(name):
        return os.environ[name]
    try:
        from google.colab import userdata
        return userdata.get(name)
    except ImportError:
        return None
    except Exception as exc:
        if type(exc).__name__ in {"SecretNotFoundError", "NotebookAccessError"}:
            return None
        raise


@lru_cache(maxsize=1)
def groq_account():
    key = optional_secret("GROQ_API_KEY")
    if not key:
        raise RuntimeError("Enable notebook access to the GROQ_API_KEY secret in Colab.")
    with OpenAI(api_key=key, base_url=GROQ_BASE, timeout=20, max_retries=0) as client:
        return key, frozenset(model.id for model in client.models.list().data)


class ModelRouting:
    def __init__(self, models, mode):
        self.models = list(models)
        self.mode = mode

    def requests(self, request):
        if request.method != "POST" or not request.url.path.endswith("/chat/completions"):
            yield None, request
            return
        body = json.loads(request.content)
        headers = {key: val for key, val in request.headers.items()
                   if key not in {"content-length", "transfer-encoding"}}
        if self.mode == "code" and body.get("tools"):
            raise ValueError("CodeAgent uses Python actions. Its API request must have no native tools.")
        for model_id in tuple(self.models):
            payload = {**body, "model": model_id}
            max_tokens = payload.pop("max_tokens", None)
            if max_tokens and "max_completion_tokens" not in payload:
                payload["max_completion_tokens"] = max_tokens
            if model_id in GPT_OSS:
                payload.pop("parallel_tool_calls", None)
                payload.pop("stop", None)
                payload["reasoning_effort"] = "low"
            elif model_id == QWEN:
                payload["reasoning_effort"] = "none"
            yield model_id, httpx.Request(
                request.method, request.url, json=payload, headers=headers,
                extensions=request.extensions,
            )

    def accept(self, model_id, status, attempt):
        if model_id is None:
            return True
        if status in {429, 503} and attempt < len(self.models):
            return False
        if 200 <= status < 300 and model_id != self.models[0]:
            self.models.remove(model_id)
            self.models.insert(0, model_id)
            print(f"Groq fallback model: {model_id}", flush=True)
        return True


class CourseTransport(httpx.HTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    def handle_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = super().handle_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            response.close()


class AsyncCourseTransport(httpx.AsyncHTTPTransport):
    def __init__(self, routing):
        super().__init__(retries=0)
        self.routing = routing

    async def handle_async_request(self, request):
        for attempt, (model_id, candidate) in enumerate(self.routing.requests(request), 1):
            response = await super().handle_async_request(candidate)
            if self.routing.accept(model_id, response.status_code, attempt):
                return response
            await response.aclose()


@lru_cache(maxsize=3)
def groq_route(mode):
    key, available = groq_account()
    candidates = {
        "text": (QWEN, *GPT_OSS),
        "vision": (QWEN,),
        "code": (QWEN, *GPT_OSS),
    }[mode]
    models = [model_id for model_id in candidates if model_id in available]
    if not models:
        raise RuntimeError(f"None of the Groq {mode} models are enabled for this project: {candidates}")
    print(f"Groq {mode} candidates, in order: {', '.join(models)}", flush=True)
    return key, ModelRouting(models, mode)


@lru_cache(maxsize=1)
def make_groq_llm():
    from llama_index.llms.openai_like import OpenAILike
    key, routing = groq_route("text")
    return OpenAILike(
        model=routing.models[0], api_base=GROQ_BASE, api_key=key,
        is_chat_model=True, is_function_calling_model=True, context_window=32768,
        max_tokens=1024, temperature=0.1, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        async_http_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=2)
def make_chat_model(mode="text"):
    from langchain_openai import ChatOpenAI
    key, routing = groq_route(mode)
    return ChatOpenAI(
        model=routing.models[0], base_url=GROQ_BASE, api_key=key,
        temperature=0.1, max_tokens=1024, timeout=30, max_retries=0,
        http_client=httpx.Client(transport=CourseTransport(routing), timeout=30),
        http_async_client=httpx.AsyncClient(transport=AsyncCourseTransport(routing), timeout=30),
    )


@lru_cache(maxsize=1)
def make_code_model():
    from smolagents import OpenAIModel
    key, routing = groq_route("code")
    return OpenAIModel(
        model_id=routing.models[0], api_base=GROQ_BASE, api_key=key,
        max_tokens=1800, temperature=0.1, retry=False,
        client_kwargs={
            "http_client": httpx.Client(transport=CourseTransport(routing), timeout=30),
            "timeout": 30, "max_retries": 0,
        },
    )

## Basic Workflow Creation

We can start by creating a simple workflow. We use the `StartEvent` and `StopEvent` classes to define the start and stop of the workflow.

In [3]:
from llama_index.core.workflow import StartEvent, StopEvent, Workflow, step


class MyWorkflow(Workflow):
    @step
    async def my_step(self, ev: StartEvent) -> StopEvent:
        # do something here
        return StopEvent(result="Hello, world!")


w = MyWorkflow(timeout=10, verbose=False)
result = await w.run()
result

'Hello, world!'

## Connecting Multiple Steps

We can also create multi-step workflows. Here we pass the event information between steps. Note that we can use type hinting to specify the event type and the flow of the workflow.

In [4]:
from llama_index.core.workflow import Event


class ProcessingEvent(Event):
    intermediate_result: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent) -> ProcessingEvent:
        # Process initial data
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

'Finished processing: Step 1 complete'

## Loops and Branches

We can also use type hinting to create branches and loops. Note that we can use the `|` operator to specify that the step can return multiple types.

In [5]:
from llama_index.core.workflow import Event
import random


class ProcessingEvent(Event):
    intermediate_result: str


class LoopEvent(Event):
    loop_output: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent | LoopEvent) -> ProcessingEvent | LoopEvent:
        if random.randint(0, 1) == 0:
            print("Bad thing happened")
            return LoopEvent(loop_output="Back to step one.")
        else:
            print("Good thing happened")
            return ProcessingEvent(intermediate_result="First step complete.")

    @step
    async def step_two(self, ev: ProcessingEvent) -> StopEvent:
        # Use the intermediate result
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(verbose=False)
result = await w.run()
result

Bad thing happened
Bad thing happened
Good thing happened


'Finished processing: First step complete.'

## Drawing Workflows

We can also draw workflows using the `draw_all_possible_flows` function.


In [6]:
from IPython.display import HTML, display
from html import escape
from pathlib import Path
from llama_index.utils.workflow import draw_all_possible_flows

diagram_path = Path("workflow.html")
draw_all_possible_flows(w, filename=str(diagram_path))
print("Workflow diagram generated:", diagram_path.name)
display(HTML(f'<iframe width="100%" height="440" srcdoc="{escape(diagram_path.read_text(encoding="utf-8"), quote=True)}"></iframe>'))

workflow.html
Workflow diagram generated: workflow.html


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


![drawing](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/unit2/llama-index/workflow-draw.png)

### State Management

Instead of passing the event information between steps, we can use the `Context` type hint to pass information between steps. 
This might be useful for long running workflows, where you want to store information between steps.

In [7]:
from llama_index.core.workflow import Event, Context
from llama_index.core.agent.workflow import ReActAgent


class ProcessingEvent(Event):
    intermediate_result: str


class MultiStepWorkflow(Workflow):
    @step
    async def step_one(self, ev: StartEvent, ctx: Context) -> ProcessingEvent:
        # Process initial data
        await ctx.store.set("query", "What is the capital of France?")
        return ProcessingEvent(intermediate_result="Step 1 complete")

    @step
    async def step_two(self, ev: ProcessingEvent, ctx: Context) -> StopEvent:
        # Use the intermediate result
        query = await ctx.store.get("query")
        print(f"Query: {query}")
        final_result = f"Finished processing: {ev.intermediate_result}"
        return StopEvent(result=final_result)


w = MultiStepWorkflow(timeout=10, verbose=False)
result = await w.run()
result

Query: What is the capital of France?


'Finished processing: Step 1 complete'

## Multi-Agent Workflows

We can also create multi-agent workflows. Here we define two agents, one that multiplies two integers and one that adds two integers.

In [8]:
from llama_index.core.agent.workflow import AgentWorkflow, FunctionAgent, ToolCallResult
from llama_index.core.agent.workflow import AgentWorkflow

# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

llm = make_groq_llm()

# we can pass functions directly without FunctionTool -- the fn/docstring are parsed for the name/description
multiply_agent = FunctionAgent(
    name="multiply_agent",
    description="Is able to multiply two integers",
    system_prompt="Use your tool to multiply numbers. For addition, hand off to add_agent.",
    tools=[multiply],
    can_handoff_to=["add_agent"], 
    llm=llm,
)

addition_agent = FunctionAgent(
    name="add_agent",
    description="Is able to add two integers",
    system_prompt="Use the add tool for each addition and report its result.",
    tools=[add], 
    llm=llm,
)

# Create the workflow
workflow = AgentWorkflow(
    agents=[multiply_agent, addition_agent],
    root_agent="multiply_agent"
)

# Run the system
handler = workflow.run(user_msg="Can you add 5 and 3?")
used = set()
async for event in handler.stream_events():
    if isinstance(event, ToolCallResult) and not event.tool_output.is_error:
        used.add(event.tool_name)
        print("Called:", event.tool_name, event.tool_kwargs, "=>", event.tool_output)
response = await handler
assert {"handoff", "add"} <= used, sorted(used)
response

Groq text candidates, in order: qwen/qwen3.8-27b, openai/gpt-oss-20b, openai/gpt-oss-120b
Groq fallback model: openai/gpt-oss-20b
Called: handoff {'reason': 'User requested addition', 'to_agent': 'add_agent'} => Agent add_agent is now handling the request due to the following reason: User requested addition.
Please continue with the current request.
Called: add {'a': 5, 'b': 3} => 8


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[ThinkingBlock(block_type='thinking', content='We need to report result.', num_tokens=None, additional_information={}), TextBlock(block_type='text', text='The result of adding 5 and 3 is **8**.')]), structured_response=None, current_agent_name='add_agent', raw={'id': 'chatcmpl-4369b611-8903-4fc0-a387-736e8f19488e', 'choices': [{'delta': {'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None}], 'model': 'openai/gpt-oss-20b', 'object': 'chat.completion.chunk', 'moderation': None, 'service_tier': None, 'system_fingerprint': 'fp_84bb35977d', 'usage': {'completion_tokens': 30, 'prompt_tokens': 345, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 7, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None, 'q